# Encrypted Quipu Test 51 — AES raw 32-byte key

AES-sealed with a random 32-byte key, paid from **apocrypha**. The key is saved so notebook 54 can release it later via a keydrop.

## How this protocol works

Same shape as notebook 50, with one difference: the AES key is **32 random bytes** generated by `secrets.token_bytes(32)`, not derived from a passphrase.

**Why this matters.** Two different access models:
- **Password (notebook 50)** — anyone with the passphrase can decrypt. Easy to share with humans but vulnerable to weak passphrases.
- **Raw key (this notebook)** — only someone with the exact 32-byte key can decrypt. The key has full entropy; it's effectively unguessable. But it has to be communicated as bytes, not as a human-rememberable string.

The raw-key form is what makes **keydrops** work. The 32 bytes can be released later via a separate keydrop inscription (notebook 54) — without that release, the content is permanently sealed.

**Protocol structure** (identical to notebook 50 except the variant byte):

```
1. inner_quipu_bytes = (inner_header, inner_body)
2. aes_key = secrets.token_bytes(32)           # NEW — random, not derived
   save to: inscriptions_ready/aes_test_key.bin
3. framed = <header_len:2 BE> + inner_header + inner_body
4. ciphertext = AES-CBC(aes_key, framed)
5. outer_header = c1dd 0001  0e  <tone>  ae  <variant=0x00>  [|TITLE|]
                                              ^
                                              00 = raw key (instead of 0x01 password)
6. body = ciphertext
```

**Fee scaling**: this notebook uses `scaled_fee()` for root and join txs (1 DOGE/KB target, floor at TIP). Each broadcast cell is followed by a wait cell that polls until ≥1 confirmation.

**Payer**: apocrypha (single-key, mi_prv).

## Setup

In [3]:
import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import os, sys, json, time, secrets
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import cryptos
import colegio_tools as ct
from colegio_tools import _txid_of_serial
from text import build_text_quipu, read_text_quipu
from encrypted import (build_aes_quipu, build_ecies_quipu, build_keydrop_quipu,
                       read_encrypted_quipu, aggregate_privkey, aggregate_pubkey,
                       TONE_ORDINARY, TONE_AFFECTION, TONE_REVERENCE)
from coincurve import PrivateKey as CCPriv, PublicKey as CCPub

doge = cryptos.Doge()
TIP_SINGLE = 5_000_000     # 0.05 DOGE per knot for single-key strand txs
TIP_MULTI  = 10_000_000    # 0.10 DOGE per knot for multisig strand txs (size-matched to 0.2 DOGE/KB)
FEE_PER_KB = 20_000_000   # 0.2 DOGE/KB target for root + join txs

def scaled_fee(draft_hex_str, floor_sat):
    """Compute fee from drafted signed-tx size at FEE_PER_KB, floor at given TIP."""
    size_bytes = len(draft_hex_str) // 2
    return max(floor_sat, (size_bytes * FEE_PER_KB) // 1000)

In [4]:
LLAVES = os.path.abspath('../../cinv/llaves')
INSCRIPTIONS_READY = os.path.join(REPO, 'inscriptions_ready')
os.makedirs(INSCRIPTIONS_READY, exist_ok=True)

def load_priv(name, password=''):
    enc = open(os.path.join(LLAVES, f'{name}_prv.enc'), 'rb').read()
    return ct.import_privKey_from_bytes(enc, password)

# This notebook uses apocrypha as payer; the AES key has no recipients.
priv_apo = load_priv('mi')
addr_apo = doge.privtoaddr(priv_apo.to_hex()[2:])
print(f'apocrypha payer: {addr_apo}')

apocrypha payer: D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX


## Generate (or load existing) AES key

In [5]:
AES_KEY_PATH = os.path.join(INSCRIPTIONS_READY, 'aes_test_key.bin')
if os.path.exists(AES_KEY_PATH):
    aes_key = open(AES_KEY_PATH, 'rb').read()
    print(f'loaded existing key from {AES_KEY_PATH}')
else:
    aes_key = secrets.token_bytes(32)
    with open(AES_KEY_PATH, 'wb') as f: f.write(aes_key)
    print(f'generated new key and saved to {AES_KEY_PATH}')
print(f'  AES key (hex): {aes_key.hex()}')

generated new key and saved to /Users/anthonyschultz/Desktop/Colegio_Invisible/inscriptions_ready/aes_test_key.bin
  AES key (hex): 427623f3ca03f89c0e8ac9e9b6849a1f1f7eb3029bd0effcc5f191351c9f5c60


## Build inner + wrap

In [6]:
inner_h, inner_b = build_text_quipu('Mensaje sellado',
                                    'Esto solo se lee con la llave de 32 bytes.',
                                    tone=TONE_AFFECTION)
outer_h, outer_b = build_aes_quipu(inner_h, inner_b, aes_key,
                                    title='cifrado', tone=TONE_ORDINARY)
print(f'outer header ({len(outer_h)} B): {outer_h.hex()}')
print(f'  byte 7 (variant) = 0x{outer_h[7]:02x} (RAW KEY, not password)')

outer header (17 B): c1dd00010e00ae007c6369667261646f7c
  byte 7 (variant) = 0x00 (RAW KEY, not password)


## Inscribe — split into strands + broadcast root

In [7]:
N_BODY_STRANDS = 4
chunk = len(outer_b) // N_BODY_STRANDS
extra = len(outer_b) %  N_BODY_STRANDS
body_parts, i = [], 0
for k in range(N_BODY_STRANDS):
    sz = chunk + (1 if k < extra else 0)
    body_parts.append(outer_b[i:i+sz]); i += sz
strand_payloads = [outer_h] + body_parts
print(f'{len(strand_payloads)} strands; sizes: {[len(p) for p in strand_payloads]}')

5 strands; sizes: [17, 25, 25, 25, 24]


In [8]:
utxos = ct.rpc_request('listunspent', [0, 9999999, [addr_apo]])
seed_inputs = [{'output': f"{u['txid']}:{u['vout']}", 'value': int(round(u['amount']*1e8))} for u in utxos]
total = sum(s['value'] for s in seed_inputs)
print(f'{len(seed_inputs)} UTXO(s), total {total/1e8:.4f} DOGE')

1 UTXO(s), total 13.3000 DOGE


### Build root tx with scaled fee

In [9]:
priv_hex = priv_apo.to_hex()[2:]
n = len(strand_payloads)

# Draft pass — placeholder seeds with TIP fee to measure tx size
draft_per = (total - TIP_SINGLE) // n
draft_seeds = [draft_per] * n
draft = doge.mktx(seed_inputs, [{'value': s, 'address': addr_apo} for s in draft_seeds])
doge.signall(draft, priv_hex)
draft_hex = cryptos.serialize(draft)
root_fee = scaled_fee(draft_hex, TIP_SINGLE)
print(f'root draft size: {len(draft_hex)//2} B  ->  scaled fee: {root_fee/1e8:.4f} DOGE')

# Real pass — redistribute strand seeds with correct fee
per = (total - root_fee) // n
remainder = (total - root_fee) - per * n
strand_seeds = [per] * n
strand_seeds[0] += remainder
root_outputs = [{'value': s, 'address': addr_apo} for s in strand_seeds]
root_tx = doge.mktx(seed_inputs, root_outputs)
doge.signall(root_tx, priv_hex)
root_hex = cryptos.serialize(root_tx)
root_txid = _txid_of_serial(root_hex)
assert ct.rpc_request('sendrawtransaction', [root_hex]) == root_txid
print(f'root_txid: {root_txid}')

root draft size: 359 B  ->  scaled fee: 0.0718 DOGE
root_txid: f7a8ee4f997f33682038192d1d378eec7260770588411bcd7fe8bebb6be3fa02


In [11]:
# Wait for root to confirm
print(f'waiting for root to confirm... ({root_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [root_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  root confs: {confs}')
        if confs >= 1:
            print(f'✓ root confirmed in block {info.get("blockhash","?")}')
            break
        start_h = h
    time.sleep(15)

waiting for root to confirm... (f7a8ee4f997f3368…)
  19:38:24  block 6213601  root confs: 6
✓ root confirmed in block 9b2bb1ec6b5f153c1e43a920c94a8ec9e9652385675d0980d7c548b903a96774


### Phase II — precompute + broadcast strand chains

In [12]:
strands = []
for si, payload in enumerate(strand_payloads):
    cad = ct.CadenaAtom(prvkey=priv_hex, data=payload,
                         utxo_dct={'output': f'{root_txid}:{si}', 'value': strand_seeds[si]},
                         tip=TIP_SINGLE)
    cad.precompute()
    strands.append(cad)
    print(f'  strand {si}: {len(cad.txns)} knots')
for si, cad in enumerate(strands):
    for hex_tx, txid in zip(cad.txns, cad.txn_ids):
        assert ct.rpc_request('sendrawtransaction', [hex_tx]) == txid
    print(f'  strand {si} broadcast')

  strand 0: 1 knots
  strand 1: 1 knots
  strand 2: 1 knots
  strand 3: 1 knots
  strand 4: 1 knots
  strand 0 broadcast
  strand 1 broadcast
  strand 2 broadcast
  strand 3 broadcast
  strand 4 broadcast


In [15]:
# Wait for all strand termini to confirm
print('waiting for strand termini to confirm...')
start_h = ct.rpc_request('getblockcount')
termini = [c.txn_ids[-1] for c in strands]
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        confs = [ct.rpc_request('getrawtransaction', [t, 1]).get('confirmations', 0) for t in termini]
        print(f'  block {h}  ' + '  '.join(f's{i}:{c}' for i,c in enumerate(confs)))
        if all(c >= 1 for c in confs):
            print('✓ all strand termini confirmed')
            break
        start_h = h
    time.sleep(15)

waiting for strand termini to confirm...
  block 6213614  s0:8  s1:8  s2:8  s3:8  s4:8
✓ all strand termini confirmed


### Phase III — build join tx with scaled fee

In [16]:
join_inputs = [{'output': f'{c.txn_ids[-1]}:0',
                'value': strand_seeds[si] - TIP_SINGLE * len(c.txns)}
               for si, c in enumerate(strands)]
join_total = sum(i['value'] for i in join_inputs)

# Draft pass
draft = doge.mktx(join_inputs, [{'value': join_total - TIP_SINGLE, 'address': addr_apo}])
doge.signall(draft, priv_hex)
draft_hex = cryptos.serialize(draft)
join_fee = scaled_fee(draft_hex, TIP_SINGLE)
print(f'join draft size: {len(draft_hex)//2} B  ->  scaled fee: {join_fee/1e8:.4f} DOGE')

# Real pass
join_tx = doge.mktx(join_inputs, [{'value': join_total - join_fee, 'address': addr_apo}])
doge.signall(join_tx, priv_hex)
join_hex = cryptos.serialize(join_tx)
join_txid = _txid_of_serial(join_hex)
assert ct.rpc_request('sendrawtransaction', [join_hex]) == join_txid
print(f'join_txid: {join_txid}')

join draft size: 943 B  ->  scaled fee: 0.1886 DOGE
join_txid: ffee5db983f37779f6b7ae801edfb0de2ba4c229128338072b58a24fd3cc7679


In [17]:
# Wait for join to confirm
print(f'waiting for join to confirm... ({join_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [join_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  join confs: {confs}')
        if confs >= 1:
            print(f'✓ join confirmed in block {info.get("blockhash","?")}')
            break
        start_h = h
    time.sleep(15)

waiting for join to confirm... (ffee5db983f37779…)
  19:51:55  block 6213616  join confs: 1
✓ join confirmed in block 4b87db46704deda73fb280f8c334d99bf9418c343008c5f12ca9087ccc312e34


## Read back from chain

In [18]:
# Walk the diamond using our local spender map
spender_map = {}
for si, cad in enumerate(strands):
    spender_map[f'{root_txid}:{si}'] = cad.txn_ids[0]
    for ki in range(len(cad.txn_ids) - 1):
        spender_map[f'{cad.txn_ids[ki]}:0'] = cad.txn_ids[ki+1]
def walk(start):
    out, cur = '', start
    while True:
        n = spender_map.get(cur)
        if not n: return out
        raw = ct.rpc_request('getrawtransaction', [n, 1])
        op = next((ct.extract_op_return(v) for v in raw['vout'] if ct.extract_op_return(v)), None)
        if not op: return out
        out += op; cur = f'{n}:0'
rec_h = bytes.fromhex(walk(f'{root_txid}:0'))
rec_b = b''.join(bytes.fromhex(walk(f'{root_txid}:{si}')) for si in range(1, len(strands)))
assert rec_h == outer_h and rec_b == outer_b
print('✓ recovered byte-identical')

✓ recovered byte-identical


## Decrypt + verify

In [19]:
parsed = read_encrypted_quipu(rec_h, rec_b, key=aes_key)
assert parsed['inner_header'] == inner_h and parsed['inner_body'] == inner_b
print('✓ decrypted byte-identical')

✓ decrypted byte-identical


In [20]:
# Show the recovered inner content
inner = read_text_quipu(parsed['inner_header'], parsed['inner_body'])
print(f'  title:  {inner["title"]!r}')
print(f'  tone:   0x{inner["tone"]:02x}')
print(f'  body:   {inner["body"]!r}')

  title:  'Mensaje sellado'
  tone:   0x01
  body:   'Esto solo se lee con la llave de 32 bytes.'


## Save manifest for notebook 54

In [21]:
json.dump({'root_txid': root_txid, 'join_txid': join_txid,
           'aes_key_path': AES_KEY_PATH, 'payer': addr_apo},
          open(os.path.join(INSCRIPTIONS_READY, 'aes_test_manifest.json'), 'w'), indent=2)
print(f'manifest saved')

manifest saved
